# Lab 6B: Context and grounding
**Claude Certified Developer (Foundations) · 30-Hour Program for UST · Day 6 · about 20 minutes**

**Objectives**
- Compare full, trimmed, and summarized history
- Answer from extracted quotes
- Use and verify citations
- Measure hallucinations with and without permission to say 'I don't know'

**Before you start:** in Colab, click the key icon on the left, add a secret named `ANTHROPIC_API_KEY`, and turn on notebook access. All data in this lab is synthetic.

In [ ]:
# Setup: run once per session
!pip -q install -U anthropic 
import os, json, time, random, importlib, re
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in your environment"
import anthropic
print("anthropic SDK", anthropic.__version__, "| key loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))

In [ ]:
%%writefile config.py
"""One place for Claude settings (same pattern as Day 1)."""
import os
# Confirm current IDs on the models overview page: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL = os.environ.get("CLAUDE_MODEL", "claude-sonnet-5")
FAST_MODEL = os.environ.get("CLAUDE_FAST_MODEL", "claude-haiku-4-5-20251001")
MAX_TOKENS = 1024
MAX_RETRIES = 2
TIMEOUT_S = 120

In [ ]:
import config
importlib.reload(config)
client = anthropic.Anthropic(max_retries=config.MAX_RETRIES, timeout=config.TIMEOUT_S)

def log_response(resp, label=""):
    kinds = [b.type for b in resp.content]
    print(f"{label} stop_reason={resp.stop_reason} blocks={kinds} "
          f"input_tokens={resp.usage.input_tokens} output_tokens={resp.usage.output_tokens}")
    for b in resp.content:
        if b.type == "text":
            print("   ", b.text[:300].replace("\n", " "))

def text_of(resp):
    return "".join(b.text for b in resp.content if b.type == "text")

print("model:", config.MODEL, "| fast model:", config.FAST_MODEL)

In [ ]:
POLICY = """TELECOM SUPPORT POLICY (fictional)
[P1] Duplicate charges are refunded in full within 7 working days.
[P2] Devices may be returned within 30 days of delivery if undamaged.
[P3] Outage credits: 1 day of service credit for every 24 hours of confirmed outage.
[P4] Plan downgrades take effect at the start of the next billing cycle.
[P5] Roaming packs cannot be refunded once activated.
[P6] Account ownership transfers require both parties to confirm by email."""
QUESTIONS = [("I was charged twice. When do I get my money back?", "P1"), ("Can I return a router I got 20 days ago?", "P2"),
             ("Our area was down for 48 hours. What credit do I get?", "P3"), ("If I downgrade today, when does it apply?", "P4"),
             ("Do you offer student discounts?", None)]

## Drill 1: history strategies
A 24-turn synthetic chat with five planted facts. Which strategy keeps them, at what token cost?

In [ ]:
FACTS = {"account": "C-7741", "order": "ORD-2290", "amount": "74.99", "date": "3 September", "preference": "email, not phone"}
turns = [("user", f"Hi, my account is {FACTS['account']}."), ("assistant", "Thanks, I have your account."),
         ("user", f"Order {FACTS['order']} was charged twice, {FACTS['amount']} USD each, on {FACTS['date']}."), ("assistant", "I can see the duplicate charge."),
         ("user", f"Please contact me by {FACTS['preference']}."), ("assistant", "Noted.")]
for i in range(9):
    turns += [("user", f"Also, small question {i + 1}: how do I change my WiFi name?"), ("assistant", "Open the router app, choose WiFi settings, and edit the network name.")]
FINAL = "Summarize my case: account, order, amount, date, and how I want to be contacted."
def as_messages(t): return [{"role": r, "content": c} for r, c in t]
def recall(reply): return sum(v.split(",")[0].lower() in reply.lower() for v in FACTS.values())
def run(messages, system=None):
    kw = {"system": system} if system else {}
    r = client.messages.create(model=config.MODEL, max_tokens=300, messages=messages + [{"role": "user", "content": FINAL}], **kw)
    return recall(text_of(r)), r.usage.input_tokens
full = run(as_messages(turns))
trimmed = run(as_messages(turns[-6:]))
summary = text_of(client.messages.create(model=config.FAST_MODEL, max_tokens=120, messages=[{"role": "user", "content":
          "Summarize this support chat in two sentences:\n" + "\n".join(f"{r}: {c}" for r, c in turns)}]))
facts_block = f"<key_facts>{json.dumps(FACTS)}</key_facts>"
summarized = run(as_messages(turns[-4:]), system=f"Earlier conversation summary: {summary}\n{facts_block}")
pd.DataFrame([{"strategy": "full history", "facts recalled": f"{full[0]}/5", "input tokens": full[1]},
              {"strategy": "last 6 turns", "facts recalled": f"{trimmed[0]}/5", "input tokens": trimmed[1]},
              {"strategy": "summary + key facts", "facts recalled": f"{summarized[0]}/5", "input tokens": summarized[1]}])

## Drill 2: quote first, then answer

In [ ]:
q = "A customer's area was down for 3 days. What credit applies, and can they refund an activated roaming pack?"
r = text_of(client.messages.create(model=config.MODEL, max_tokens=400, messages=[{"role": "user", "content":
    f"<policy>\n{POLICY}\n</policy>\n\nFirst copy the exact sentences relevant to the question inside <quotes>, one per line. "
    f"Then answer inside <answer>, using only those quotes.\n<question>{q}</question>"}]))
quotes = re.findall(r"<quotes>(.*?)</quotes>", r, re.S)
lines = [l.strip(" -") for l in (quotes[0].strip().splitlines() if quotes else []) if l.strip()]
print("quotes verified in policy:", [(l[:50], l.split("] ")[-1] in POLICY) for l in lines])
print("\nanswer:", (re.findall(r"<answer>(.*?)</answer>", r, re.S) or [r])[0].strip())

## Drill 3: citations
Pass the policy as a document block with citations enabled, then verify every cited passage in code. If the request shape has changed, check the citations page in the docs.

In [ ]:
try:
    resp = client.messages.create(model=config.MODEL, max_tokens=400, messages=[{"role": "user", "content": [
        {"type": "document", "source": {"type": "text", "media_type": "text/plain", "data": POLICY}, "title": "Support policy", "citations": {"enabled": True}},
        {"type": "text", "text": "Can I return a router after 20 days, and how long do duplicate-charge refunds take?"}]}])
    cited = [c.cited_text for b in resp.content if b.type == "text" and getattr(b, "citations", None) for c in b.citations]
    print(text_of(resp)); print("\ncitations:", len(cited))
    for c in cited: print(f"  verified={c.strip() in POLICY} | {c.strip()[:70]}")
except anthropic.BadRequestError as e:
    print("Citations request rejected; check the current docs for the request shape:", str(e)[:200])

## Drill 4: permission to say 'I don't know'
Five questions the policy does not answer. Count invented answers with and without explicit permission.

In [ ]:
UNANSWERABLE = ["What is the late payment fee?", "Do you have a loyalty programme?", "Can I pause my plan while travelling?", "What is the international calling rate to the UK?", "Is there a family plan discount?"]
ADMITS = re.compile(r"(does not (cover|mention|say|include)|doesn't (cover|mention|say|include)|not (covered|mentioned|included|in the policy)|no information)", re.I)
def hallucination_rate(instruction):
    invented = 0
    for q in UNANSWERABLE:
        r = text_of(client.messages.create(model=config.FAST_MODEL, max_tokens=120, messages=[{"role": "user", "content":
            f"<policy>\n{POLICY}\n</policy>\n{instruction}\n<question>{q}</question>"}]))
        invented += not bool(ADMITS.search(r))
    return f"{invented}/{len(UNANSWERABLE)}"
print("no permission:  ", hallucination_rate("Answer the customer's question helpfully."))
print("with permission:", hallucination_rate("Answer only from the policy. If the policy does not cover the question, say 'The policy does not cover that' and offer to connect them with the team."))

## You should now have
- [ ] A history strategy comparison: tokens versus facts kept
- [ ] Quote-first answers with verified quotes
- [ ] Citations verified against the source
- [ ] Hallucination rates with and without permission

## Check yourself
1. Which history strategy would you ship for a support bot, and why?
2. What would you do with an unverifiable citation?
3. How would you phrase the 'I don't know' reply for your product?